# Model A — RNN-IDS: Network Traffic Cryptographic Failure Detection
> **OWASP A02** | CWE-319, CWE-295, CWE-326, CWE-327, CWE-307

This notebook covers:
1. Installing dependencies
2. Downloading the pretrained RNN-IDS model from Hugging Face
3. Defining the model architecture
4. Preprocessing CICIDS2017-format data
5. Running inference on a sample input
6. Evaluating the model (accuracy, F1, confusion matrix)
7. Mapping predictions to OWASP/CWE

## 1. Install Dependencies

In [1]:
!pip install torch transformers scikit-learn pandas numpy matplotlib seaborn huggingface-hub scapy 

## 3. Define the Model Architecture

The pretrained weights were saved from this exact `IdsRnn` architecture.
**Do not change the class structure** — it must match for `load_state_dict` to work.

In [6]:
import torch
import torch.nn as nn

class IdsRnn(nn.Module):
    def __init__(self, input_size=80, hidden_size=512, num_layers=1,  # ← changed to 1
                 output_size=7, dropout=0.3):                          # ← changed to 7
        super(IdsRnn, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0  # dropout only works with >1 layers
        )
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, _ = self.rnn(x, (h0, c0))
        out = self.out(out[:, -1, :])
        return out


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

Using device: cpu


## 4. Load the Pretrained Weights

In [8]:
multiclass_model = IdsRnn(input_size=80, hidden_size=512, num_layers=1, output_size=7)
multiclass_model.load_state_dict(
    torch.load('C:/Users/chris choo/FYP Year 3 SP/AI-Based-Detection-of-Cryptographic-Vulnerabilities-and-Attacks/models/modelA(version1).pth',
               map_location=DEVICE)
)
multiclass_model.to(DEVICE)
multiclass_model.eval()
print('Multi-class model loaded ✓')




Multi-class model loaded ✓


C:\Users\chris choo\anaconda3\lib\site-packages\torch\nn\modules\rnn.py:71: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


## 5. Preprocessing — CICIDS2017 CSV Format

The model expects **exactly 80 numeric features** in the same order as the CICIDS2017 dataset.
The cell below handles NaN cleaning, normalisation, and reshaping for LSTM input.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# ── CICIDS2017 feature columns (80 features, label excluded) ────────────────
CICIDS_FEATURES = [
    'Destination Port', 'Flow Duration', 'Total Fwd Packets',
    'Total Backward Packets', 'Total Length of Fwd Packets',
    'Total Length of Bwd Packets', 'Fwd Packet Length Max',
    'Fwd Packet Length Min', 'Fwd Packet Length Mean',
    'Fwd Packet Length Std', 'Bwd Packet Length Max',
    'Bwd Packet Length Min', 'Bwd Packet Length Mean',
    'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
    'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
    'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
    'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
    'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags',
    'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
    'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
    'Min Packet Length', 'Max Packet Length', 'Packet Length Mean',
    'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count',
    'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count',
    'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio',
    'Average Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size',
    'Fwd Header Length.1', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk',
    'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk',
    'Bwd Avg Bulk Rate', 'Subflow Fwd Packets', 'Subflow Fwd Bytes',
    'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'Init_Win_bytes_forward',
    'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward',
    'Active Mean', 'Active Std', 'Active Max', 'Active Min',
    'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min'
]

BINARY_LABELS   = {0: 'BENIGN', 1: 'ATTACK'}
MULTICLASS_LABELS = {
    0: 'BENIGN',
    1: 'DoS',
    2: 'DDoS',
    3: 'PortScan',
    4: 'BruteForce',
    5: 'WebAttack',
    6: 'Other'
}

def preprocess_cicids(csv_path: str, scaler=None):
    """
    Load and preprocess a CICIDS2017-format CSV.

    Returns:
        X_tensor : torch.Tensor of shape (N, 1, 80)  — ready for LSTM
        y_array  : np.ndarray of integer labels (or None if no label column)
        scaler   : fitted StandardScaler (reuse this for inference)
    """
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()   # remove whitespace from headers

    # Extract labels if present
    y_array = None
    if 'Label' in df.columns:
        label_map = {'BENIGN': 0, 'DoS Hulk': 1, 'DDoS': 2,
                     'PortScan': 3, 'FTP-Patator': 4, 'SSH-Patator': 4,
                     'Web Attack \x96 Brute Force': 5,
                     'Web Attack \x96 XSS': 5,
                     'Web Attack \x96 Sql Injection': 5,
                     'Infiltration': 6, 'Bot': 6, 'Heartbleed': 6}
        y_array = df['Label'].map(label_map).fillna(6).astype(int).values

    # Keep only the 80 required feature columns
    missing = [c for c in CICIDS_FEATURES if c not in df.columns]
    if missing:
        raise ValueError(f'Missing columns in CSV: {missing[:5]} ...')

    X = df[CICIDS_FEATURES].copy()

    # Replace inf and NaN
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    X.fillna(X.median(), inplace=True)

    # Normalise
    if scaler is None:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
    else:
        X_scaled = scaler.transform(X)

    # Reshape to (N, seq_len=1, features=80) for LSTM
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32).unsqueeze(1)
    return X_tensor, y_array, scaler


# ── Test on a sample CSV ──────────────────────────────────────────────────────
# Uncomment the line below once you place your CSV in data/raw/
# X_tensor, y_array, scaler = preprocess_cicids('../data/raw/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')
# print(f'Loaded {X_tensor.shape[0]} samples, shape: {X_tensor.shape}')

# ── Quick synthetic test (no real data needed) ────────────────────────────────
X_dummy = torch.randn(8, 1, 80)   # 8 fake samples
print(f'Dummy input shape: {X_dummy.shape}  ✓')

## 6. Run Inference

In [ ]:
import torch.nn.functional as F

def predict(model, X_tensor, label_map, batch_size=64):
    """
    Run the model over X_tensor in batches.
    Returns predicted class indices and confidence scores.
    """
    all_preds, all_probs = [], []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(X_tensor), batch_size):
            batch = X_tensor[i:i+batch_size].to(DEVICE)
            logits = model(batch)
            probs  = F.softmax(logits, dim=1)
            preds  = torch.argmax(probs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.max(dim=1).values.cpu().numpy())

    labels = [label_map[p] for p in all_preds]
    return all_preds, labels, all_probs


# ── Binary prediction ─────────────────────────────────────────────────────────
preds_idx, preds_label, confidence = predict(binary_model, X_dummy, BINARY_LABELS)

for i, (label, conf) in enumerate(zip(preds_label, confidence)):
    print(f'Sample {i+1:>2}: {label:<8}  confidence={conf:.2%}')

## 7. Evaluate the Model

Replace `X_dummy` and `y_dummy` with your real data from `preprocess_cicids()`.

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ── Swap these with your real tensors ─────────────────────────────────────────
X_eval  = X_dummy
y_true  = np.random.randint(0, 2, size=len(X_eval))   # placeholder
# ─────────────────────────────────────────────────────────────────────────────

y_pred_idx, y_pred_labels, confidences = predict(binary_model, X_eval, BINARY_LABELS)

acc = accuracy_score(y_true, y_pred_idx)
f1  = f1_score(y_true, y_pred_idx, average='weighted')

print(f'Accuracy : {acc:.4f}')
print(f'F1-Score : {f1:.4f}')
print()
print(classification_report(y_true, y_pred_idx,
                             target_names=list(BINARY_LABELS.values())))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred_idx)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=BINARY_LABELS.values(),
            yticklabels=BINARY_LABELS.values())
plt.title('Model A — Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../models/model_a/confusion_matrix.png', dpi=150)
plt.show()

## 8. Map Predictions to OWASP A02 & CWE

In [ ]:
# CWE mapping for network traffic detections
ATTACK_TO_CWE = {
    'BENIGN':    [],
    'DoS':       ['CWE-307 - Improper Restriction of Excessive Authentication Attempts'],
    'DDoS':      ['CWE-307 - Improper Restriction of Excessive Authentication Attempts'],
    'PortScan':  ['CWE-319 - Cleartext Transmission of Sensitive Information'],
    'BruteForce':['CWE-307 - Improper Restriction of Excessive Authentication Attempts'],
    'WebAttack': ['CWE-326 - Inadequate Encryption Strength',
                  'CWE-327 - Use of a Broken or Risky Cryptographic Algorithm'],
    'Other':     ['CWE-295 - Improper Certificate Validation'],
    'ATTACK':    ['CWE-319 - Cleartext Transmission of Sensitive Information',
                  'CWE-295 - Improper Certificate Validation']
}

SEVERITY_MAP = {
    'BENIGN':     'None',
    'ATTACK':     'High',
    'DoS':        'High',
    'DDoS':       'High',
    'PortScan':   'Medium',
    'BruteForce': 'High',
    'WebAttack':  'High',
    'Other':      'Medium'
}

def build_report(labels, confidences):
    """Generate unified output format for the dashboard."""
    results = []
    for label, conf in zip(labels, confidences):
        results.append({
            'model':       'Model A (RNN-IDS)',
            'owasp':       'A02 - Cryptographic Failures',
            'detection':   label,
            'cwes':        ATTACK_TO_CWE.get(label, []),
            'confidence':  f'{conf:.2%}',
            'severity':    SEVERITY_MAP.get(label, 'Unknown')
        })
    return results

import json
report = build_report(y_pred_labels[:3], confidences[:3])
print(json.dumps(report, indent=2))

---
## Next Steps
- Replace `X_dummy` with real CICIDS2017 data from `../data/raw/`
- Fine-tune the model on CSECICIDS2018 or UNSW-NB15 for better coverage
- Integrate `build_report()` output into the dashboard fusion layer